# 01 — Comprensión de los datos

**Objetivo:** conocer los dos CSV *antes* de tocarlos. En este notebook **solo observamos**:
no limpiamos ni corregimos nada. Lo que encontremos va a `docs/02_diccionario.md`.

Archivos:
- `freMTPL2freq.csv` → una fila por **póliza**.
- `freMTPL2sev.csv` → una fila por **siniestro**.

In [1]:
import pandas as pd

# Mostrar todas las columnas y números sin notación científica
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

freq = pd.read_csv("../datasets/freMTPL2freq.csv")
sev = pd.read_csv("../datasets/freMTPL2sev.csv")

## 1. Forma: ¿cuántas filas y columnas?

In [2]:
print("freq:", freq.shape)
print("sev: ", sev.shape)

freq: (678013, 12)
sev:  (26639, 2)


In [ ]:
freq.head()

IDPol       ClaimNb

375159       5

,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region
0,1.0000,1,0.1000,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
1,3.0000,1,0.7700,5,0,55,50,B12,Regular,D,1217,Rhone-Alpes
2,5.0000,1,0.7500,6,2,52,50,B12,Diesel,B,54,Picardie
3,10.0000,1,0.0900,7,0,46,50,B12,Diesel,B,76,Aquitaine
4,11.0000,1,0.8400,7,0,46,50,B12,Diesel,B,76,Aquitaine


In [ ]:
sev.head()


IDPol       ClaimNb

375159       124
375159       123
375159        58
375159        41 


,IDpol,ClaimAmount
0,1552,995.2000
1,1010996,"1,128.1200"
2,4024277,"1,851.1100"
3,4007252,"1,204.0000"
4,4046424,"1,204.0000"


## 2. Tipos de datos

Ojo con `IDpol`: es un identificador, debería ser entero. Veamos cómo lo lee pandas en cada archivo.

In [5]:
freq.dtypes

IDpol         float64
ClaimNb         int64
Exposure      float64
VehPower        int64
VehAge          int64
DrivAge         int64
BonusMalus      int64
VehBrand          str
VehGas            str
Area              str
Density         int64
Region            str
dtype: object

In [6]:
sev.dtypes

IDpol            int64
ClaimAmount    float64
dtype: object

`IDpol` se lee como **float** en `freq` y como **int** en `sev`. ¿Hay algún IDpol con decimales
de verdad? Si no hay, el float es solo una cuestión de formato del archivo (ej. `1.0`).

In [7]:
# Cantidad de IDpol en freq con parte decimal distinta de cero
(freq["IDpol"] % 1 != 0).sum()

np.int64(0)

No hay decimales reales. Entonces, ¿por qué pandas eligió float? Leemos `IDpol` **como texto**
y buscamos valores que no sean solo dígitos.

In [8]:
ids_texto = pd.read_csv("../datasets/freMTPL2freq.csv", usecols=["IDpol"], dtype=str)["IDpol"]
raros = ids_texto[~ids_texto.str.fullmatch(r"[0-9]+")]
print("IDpol que no son un entero 'limpio':", len(raros))
raros

IDpol que no son un entero 'limpio': 1


49331    1.00E+05
Name: IDpol, dtype: str

In [9]:
# Miramos las filas vecinas para entender qué valor debería ser
freq.iloc[raros.index[0] - 2 : raros.index[0] + 3]

,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Area,Density,Region
49329,"99,998.0000",0,0.9100,7,7,44,50,B1,Diesel,D,1217,Rhone-Alpes
49330,"99,999.0000",1,0.8300,7,6,25,118,B1,Diesel,E,9307,Rhone-Alpes
49331,"100,000.0000",0,0.9200,7,10,24,85,B3,Regular,D,1871,Rhone-Alpes
49332,"100,001.0000",0,1.0000,4,12,36,50,B6,Regular,C,141,Rhone-Alpes
49333,"100,003.0000",0,0.9200,7,12,27,80,B1,Regular,E,2472,Rhone-Alpes


Hay **una** fila con `IDpol = "1.00E+05"` (notación científica, típico de un archivo que pasó por
Excel). Por esa única fila pandas lee toda la columna como float. Por las filas vecinas
(99999 y 100001) el valor sería **100000**. *No lo corregimos acá*: queda anotado para la limpieza.

## 3. Nulos

In [10]:
freq.isna().sum()

IDpol         0
ClaimNb       0
Exposure      0
VehPower      0
VehAge        0
DrivAge       0
BonusMalus    0
VehBrand      0
VehGas        0
Area          0
Density       0
Region        0
dtype: int64

In [11]:
sev.isna().sum()

IDpol          0
ClaimAmount    0
dtype: int64

## 4. Rangos de las variables numéricas

Mostramos mínimo, máximo y algunos percentiles. Buscamos valores raros (ej. edades de 100 años,
exposición mayor a 1 año).

In [12]:
percentiles = [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
freq.describe(percentiles=percentiles).T

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
IDpol,"678,013.0000","2,621,856.9211","1,641,782.7527",1.0000,"19,934.2400","69,365.6000","1,157,951.0000","2,272,152.0000","4,046,274.0000","6,014,195.2000","6,105,127.7600","6,114,330.0000"
ClaimNb,"678,013.0000",0.0532,0.2401,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,1.0000,16.0000
Exposure,"678,013.0000",0.5288,0.3644,0.0027,0.0082,0.0400,0.1800,0.4900,0.9900,1.0000,1.0000,2.0100
VehPower,"678,013.0000",6.4546,2.0509,4.0000,4.0000,4.0000,5.0000,6.0000,7.0000,11.0000,13.0000,15.0000
VehAge,"678,013.0000",7.0443,5.6662,0.0000,0.0000,0.0000,2.0000,6.0000,11.0000,17.0000,21.0000,100.0000
DrivAge,"678,013.0000",45.4991,14.1374,18.0000,20.0000,25.0000,34.0000,44.0000,55.0000,72.0000,80.0000,100.0000
BonusMalus,"678,013.0000",59.7615,15.6367,50.0000,50.0000,50.0000,50.0000,50.0000,64.0000,95.0000,106.0000,230.0000
Density,"678,013.0000","1,792.4224","3,958.6466",1.0000,10.0000,20.0000,92.0000,393.0000,"1,658.0000","7,313.0000","27,000.0000","27,000.0000"


In [13]:
sev.describe(percentiles=percentiles).T

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
IDpol,"26,639.0000","2,279,863.8311","1,577,201.8069",139.0000,"23,196.3200","58,882.8000","1,087,642.5000","2,137,413.0000","3,180,162.0000","5,056,765.8000","6,093,356.7200","6,113,971.0000"
ClaimAmount,"26,639.0000","2,278.5364","29,297.4813",1.0000,40.0314,79.1230,686.8100,"1,172.0000","1,228.0800","4,861.6850","16,793.7044","4,075,400.5600"


## 5. Variables categóricas

Valores únicos y cuántas pólizas hay en cada uno.

In [14]:
for col in ["Area", "VehGas", "VehBrand"]:
    print(f"--- {col} ({freq[col].nunique()} valores) ---")
    print(freq[col].value_counts().sort_index(), "\n")

--- Area (6 valores) ---
Area
A    103957
B     75459
C    191880
D    151596
E    137167
F     17954
Name: count, dtype: int64 

--- VehGas (2 valores) ---
VehGas
Diesel     332136
Regular    345877
Name: count, dtype: int64 

--- VehBrand (11 valores) ---
VehBrand
B1     162736
B10     17707
B11     13585
B12    166024
B13     12178
B14      4047
B2     159861
B3      53395
B4      25179
B5      34753
B6      28548
Name: count, dtype: int64 



### Region

Teníamos la hipótesis de que `Region` venía como **código INSEE** (ej. `11` = Île-de-France).
Veamos los valores reales.

In [15]:
print("Valores únicos:", freq["Region"].nunique())
freq["Region"].value_counts()

Valores únicos: 21


Region
Centre                         160601
Rhone-Alpes                     84752
Provence-Alpes-Cotes-D'Azur     79315
Ile-de-France                   69791
Bretagne                        42122
Nord-Pas-de-Calais              40275
Pays-de-la-Loire                38751
Languedoc-Roussillon            35805
Aquitaine                       31329
Poitou-Charentes                19046
Midi-Pyrenees                   17141
Basse-Normandie                 10893
Bourgogne                       10492
Haute-Normandie                  8784
Picardie                         7994
Auvergne                         5287
Limousin                         4567
Corse                            4516
Champagne-Ardenne                3026
Alsace                           2200
Franche-Comte                    1326
Name: count, dtype: int64

## 6. Claves: ¿IDpol es único?

- En `freq` esperamos **una fila por póliza** → filas = IDpol únicos.
- En `sev` esperamos **varias filas por póliza** (una por siniestro) → filas > IDpol únicos.

In [16]:
resumen = pd.DataFrame({
    "filas": [len(freq), len(sev)],
    "IDpol_unicos": [freq["IDpol"].nunique(), sev["IDpol"].nunique()],
}, index=["freq", "sev"])
resumen["filas_repetidas"] = resumen["filas"] - resumen["IDpol_unicos"]
resumen

,filas,IDpol_unicos,filas_repetidas
freq,678013,678013,0
sev,26639,24950,1689


In [17]:
# ¿Cuántos siniestros tiene cada póliza en sev? (distribución)
sev.groupby("IDpol").size().value_counts().sort_index()

1     23571
2      1298
3        62
4         5
5         2
6         1
8         1
9         1
11        2
16        1
21        1
23        1
24        1
25        1
36        1
66        1
Name: count, dtype: int64

## 7. Primeras sospechas (solo contamos, no corregimos)

Estas cuentas las vamos a repetir en SQL (ver docs/04_hallazgos_calidad.md). Acá sirven para anotar sospechas.

In [18]:
ids_freq = set(freq["IDpol"])
huerfanos = sev[~sev["IDpol"].isin(ids_freq)]
print("Siniestros en sev sin póliza en freq:", len(huerfanos))
print("...y pertenecen a esta cantidad de IDpol distintos:", huerfanos["IDpol"].nunique())

Siniestros en sev sin póliza en freq: 195
...y pertenecen a esta cantidad de IDpol distintos: 6


In [19]:
# Comparamos ClaimNb (lo que dice freq) contra la cantidad de filas en sev para cada póliza
n_sev = sev.groupby("IDpol").size().rename("n_sev")
comp = freq[["IDpol", "ClaimNb"]].merge(n_sev, left_on="IDpol", right_index=True, how="left")
comp["n_sev"] = comp["n_sev"].fillna(0).astype(int)
distintas = comp[comp["ClaimNb"] != comp["n_sev"]]
print("Pólizas con ClaimNb distinto a la cantidad de siniestros en sev:", len(distintas))
print("...de esas, con IDpol <= 24500:", (distintas["IDpol"] <= 24500).sum())

Pólizas con ClaimNb distinto a la cantidad de siniestros en sev: 9117
...de esas, con IDpol <= 24500: 8851


In [20]:
# Cuentas rápidas de rangos sospechosos
print("Exposure > 1:          ", (freq["Exposure"] > 1).sum())
print("VehAge > 40:           ", (freq["VehAge"] > 40).sum())
print("DrivAge < 18:          ", (freq["DrivAge"] < 18).sum())
print("BonusMalus fuera 50-350:", ((freq["BonusMalus"] < 50) | (freq["BonusMalus"] > 350)).sum())
print("ClaimAmount <= 0:      ", (sev["ClaimAmount"] <= 0).sum())

Exposure > 1:           1224
VehAge > 40:            230
DrivAge < 18:           0
BonusMalus fuera 50-350: 0
ClaimAmount <= 0:       0


In [21]:
# ¿Las zonas (Area) se ordenan por densidad?
freq.groupby("Area")["Density"].agg(["min", "median", "max", "count"])

,min,median,max,count
Area,,,,
A,1,27.0000,50,103957
B,50,72.0000,100,75459
C,100,222.0000,500,191880
D,500,"1,054.0000",1993,151596
E,2001,"3,744.0000",9850,137167
F,10008,"27,000.0000",27000,17954


## Conclusión del notebook

Lo anotado acá (tipos, rangos, valores únicos, claves y sospechas) se resume en
`docs/02_diccionario.md`. **No modificamos ningún dato.**